# Урок 11 - Протокол «Агент до агента» (A2A)


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Що таке протокол A2A?

**Протокол «Агент до Агента» (A2A)** — це відкритий стандарт, який дозволяє AI-агентам спілкуватися,
знаходити один одного та співпрацювати — навіть якщо вони створені на різних платформах або розміщені
на різних сервісах.

Основні концепції:

- **Виявлення** — агенти публікують *Картку агента*, яка описує їхні можливості, що
  полегшує іншим агентам (або оркестраторам) знаходити потрібного спеціаліста для виконання завдання.
- **Обмін повідомленнями** — агенти обмінюються структурованими повідомленнями через спільний протокол, тож
  запит від одного агента може бути зрозумілим і виконаним іншим незалежно від внутрішньої
  реалізації.
- **Життєвий цикл завдання** — A2A визначає стани, такі як *надіслано*, *в роботі*, *завершено* та
  *не вдалося*, даючи оркестратору повний огляд того, як просувається делеговане завдання.

У цьому уроці ми імітуємо співпрацю в стилі A2A, підключаючи трьох спеціалізованих туристичних агентів
до робочого процесу, у якому кожен агент вносить свій експертний внесок і передає результати наступному.


## Створення спеціалізованих туристичних агентів


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Співпраця кількох агентів через робочий процес

Ми підключаємо трьох агентів у послідовний робочий процес, який відтворює передачу повідомлень A2A:

1. **CurrencyExchangeAgent** отримує запит користувача і надає поради щодо валюти.
2. **ActivityPlannerAgent** отримує збагачений контекст і додає рекомендації щодо діяльності.
3. **TravelManagerAgent** синтезує обидва вхідні дані у підсумковий туристичний брифінг.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Розуміння A2A у виробництві

У виробничому середовищі протокол A2A відкриває потужні сценарії взаємодії між сервісами:

| Можливість | Опис |
|---|---|
| **Взаємодія між фреймворками** | Агент, створений на одному фреймворку, може делегувати завдання агенту, створеному на будь-якому іншому сумісному з A2A фреймворку, що забезпечує справжню міжорганізаційну взаємодію. |
| **Межі сервісів** | Агенти можуть знаходитись у різних мікросервісах, хмарних регіонах або навіть в різних організаціях, при цьому безперешкодно співпрацюючи. |
| **Динамічне виявлення** | Оркестратор може запитувати реєстр Agent Card під час виконання, щоб знайти найкращого спеціаліста для певного підзавдання. |
| **Потокова передача та push-повідомлення** | A2A підтримує Server-Sent Events (SSE) для оновлення прогресу в реальному часі та push-повідомлення для довготривалих завдань. |

Потік робіт, який ми побудували вище, є спрощеною, внутрішньопроцесною версією цього патерну. У реальному
розгортанні кожен агент би відкривав HTTP-ендпоінт, публікував Agent Card та спілкувався
через протокол A2A JSON-RPC.


## Підсумок

У цьому уроці ви дізналися:

1. **Що таке протокол A2A** — відкритий стандарт для виявлення, обміну повідомленнями
   та керування завданнями між агентами.
2. **Як створювати спеціалізованих агентів** — агента обміну валюти, агента планування діяльності
   та організатора Travel Manager.
3. **Як пов’язувати агентів у робочий процес** — використовуючи `WorkflowBuilder` для моделювання послідовної
   передачі повідомлень між агентами.
4. **Як A2A працює у виробничому середовищі** — забезпечення співпраці між різними фреймворками та сервісами
   з динамічним виявленням і потоковими оновленнями.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
